In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import (
    XGBClassifier,
    XGBRegressor
)

In [2]:
df = pd.read_csv("../data/processed/merged_dataset.csv")

print(df.shape)
df.head()

(27422, 44)


,designation,full_name,close_approach_date,distance_au,dist_km,dist_lunar,distance_min_au,distance_max_au,velocity_km_s,v_rel_kmh,...,first_obs,last_obs,data_arc,data_arc_years,orbit_stretch,interaction_index,size_energy_index,diameter_velocity_proxy,year,month
0,2022 AP1,(2022 AP1),2015-01-01 00:27:00,0.045100,6746806.0,17.55,0.013778,0.076571,11.891749,42810.0,...,2022-01-04,2022-01-09,5.0,0.01,0.656,1112.347052,0.000056,0.088656,2015,1
1,2015 AE45,(2015 AE45),2015-01-02 15:56:00,0.048239,7216474.0,18.77,0.048220,0.048258,7.065686,25436.0,...,2015-01-15,2015-02-19,35.0,0.10,0.827,30.863245,0.000957,0.218582,2015,1
2,613286,613286 (2005 YQ96),2015-01-02 21:46:00,0.026522,3967664.0,10.32,0.026522,0.026522,12.703378,45732.0,...,2005-12-30,2023-12-15,6559.0,17.96,0.495,52.080621,0.070619,3.375827,2015,1
3,2014 YQ34,(2014 YQ34),2015-01-03 13:29:00,0.079692,11921722.0,31.01,0.078275,0.081108,24.982094,89936.0,...,2014-12-23,2014-12-29,6.0,0.02,4.194,156.715248,0.002735,1.306441,2015,1
4,2014 YE42,(2014 YE42),2015-01-03 15:00:00,0.010995,1644896.0,4.28,0.010971,0.011020,13.998882,50396.0,...,2014-12-30,2015-01-04,5.0,0.01,2.839,698.812020,0.005507,1.038854,2015,1


In [3]:
FEATURES = [
    "diameter_km",
    "H",
    "e",
    "a",
    "i",
    "q",
    "ad",
    "per_y",
    "moid_au",
    "n",
    "condition_code",
    "orbit_stretch",
    "interaction_index",
    "size_energy_index",
    "diameter_velocity_proxy"
]

In [4]:
train_df = df[df["is_future"] == False].copy()

test_df = df[df["is_future"] == True].copy()

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 23537
Test: 3885


In [5]:
"pha"

'pha'

In [6]:
X_train = train_df[FEATURES]
X_test = test_df[FEATURES]

y_train = train_df["pha"].astype(int)
y_test = test_df["pha"].astype(int)

In [7]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print(scale_pos_weight)

26.24189814814815


In [8]:
preprocessor = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    )
                ]
            ),
            FEATURES
        )
    ]
)

In [9]:
hazard_model = Pipeline(
    [
        ("prep", preprocessor),

        (
            "model",
            XGBClassifier(
                n_estimators=500,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

In [10]:
hazard_model.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['diameter_km','H','e',...,'interaction_index','size_energy_index', 'diameter_velocity_proxy']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all

In [11]:
hazard_pred = hazard_model.predict(
    X_test
)

hazard_prob = hazard_model.predict_proba(
    X_test
)[:,1]

In [12]:
print(
    "Accuracy:",
    accuracy_score(
        y_test,
        hazard_pred
    )
)

print(
    "ROC AUC:",
    roc_auc_score(
        y_test,
        hazard_prob
    )
)

print(
    classification_report(
        y_test,
        hazard_pred
    )
)

Accuracy: 0.9969111969111969
ROC AUC: 0.9991516784814576
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3380
           1       0.99      0.98      0.99       505

    accuracy                           1.00      3885
   macro avg       0.99      0.99      0.99      3885
weighted avg       1.00      1.00      1.00      3885



In [13]:
"distance_au"

'distance_au'

In [14]:
distance_model = Pipeline(
    [
        ("prep", preprocessor),

        (
            "model",
            XGBRegressor(
                n_estimators=500,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42
            )
        )
    ]
)

In [15]:
distance_model.fit(
    train_df[FEATURES],
    train_df["distance_au"]
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['diameter_km','H','e',...,'interaction_index','size_energy_index', 'diameter_velocity_proxy']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of column

In [16]:
distance_pred = distance_model.predict(
    test_df[FEATURES]
)

In [17]:
print(
    "MAE:",
    mean_absolute_error(
        test_df["distance_au"],
        distance_pred
    )
)

print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(
            test_df["distance_au"],
            distance_pred
        )
    )
)

print(
    "R2:",
    r2_score(
        test_df["distance_au"],
        distance_pred
    )
)

MAE: 0.02448549432267789
RMSE: 0.03153928724294502
R2: -0.4940621801379692


In [18]:
"velocity_km_s"

'velocity_km_s'

In [19]:
velocity_model = Pipeline(
    [
        ("prep", preprocessor),

        (
            "model",
            XGBRegressor(
                n_estimators=500,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42
            )
        )
    ]
)

In [20]:
velocity_model.fit(
    train_df[FEATURES],
    train_df["velocity_km_s"]
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['diameter_km','H','e',...,'interaction_index','size_energy_index', 'diameter_velocity_proxy']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of column

In [21]:
velocity_pred = velocity_model.predict(
    test_df[FEATURES]
)

In [22]:
print(
    "MAE:",
    mean_absolute_error(
        test_df["velocity_km_s"],
        velocity_pred
    )
)

print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(
            test_df["velocity_km_s"],
            velocity_pred
        )
    )
)

print(
    "R2:",
    r2_score(
        test_df["velocity_km_s"],
        velocity_pred
    )
)

MAE: 0.6395158781400541
RMSE: 0.8696794703461541
R2: 0.9740006403060236


In [23]:
simulation_df = test_df.copy()

simulation_df[
    "hazard_probability"
] = hazard_prob

simulation_df[
    "predicted_distance_au"
] = distance_pred

simulation_df[
    "predicted_velocity_km_s"
] = velocity_pred

In [24]:
simulation_df["risk_score"] = (
    simulation_df["diameter_km"]
    *
    simulation_df["predicted_velocity_km_s"]
) / (
    simulation_df["predicted_distance_au"]
    + 1e-6
)

In [25]:
simulation_df = simulation_df.sort_values(
    "risk_score",
    ascending=False
)

In [26]:
OUTPUT_COLUMNS = [

    "spkid",

    "full_name",

    "hazard_probability",

    "predicted_distance_au",

    "predicted_velocity_km_s",

    "risk_score",

    "a",
    "e",
    "i",
    "q",
    "ad",
    "per_y",

    "diameter_km"
]

In [27]:
simulation_df[
    OUTPUT_COLUMNS
].to_csv(
    "../data/exports/asteroid_predictions.csv",
    index=False
)

In [28]:
simulation_df[
    OUTPUT_COLUMNS
].to_json(
    "../data/exports/asteroid_predictions.json",
    orient="records",
    indent=2
)

In [29]:
joblib.dump(
    hazard_model,
    "../data/exports/hazard_model.pkl"
)

joblib.dump(
    distance_model,
    "../data/exports/distance_model.pkl"
)

joblib.dump(
    velocity_model,
    "../data/exports/velocity_model.pkl"
)

['../data/exports/velocity_model.pkl']